# Análisis F1 — Feature Engineering y Correlaciones

Limpieza, generación de features y análisis de correlación con `RACE_Position` sobre `f1_all_full.csv`.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv("../include/output/f1_all_full.csv", sep=";")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 1. Datos de prácticas (unificados y con diffs desde silver)

El CSV `f1_all_full.csv` ya viene con columnas `FP_...` unificadas (sin división por sesión) y con `_DBT_%` calculada por carrera.

In [ ]:
df_unified = df.copy()

In [ ]:
df_unified.info()

## 2. Features de posición y tiempos de clasificación

In [ ]:
df_unified['Position_Gain_From_Grid'] = df_unified['Q_GridPosition'] - df_unified['RACE_Position']

# Sort by DriverId, Year, and RoundNumber to ensure correct cumulative average calculation
df_unified = df_unified.sort_values(by=['DriverId', 'Year', 'RoundNumber'])

# Calculate the cumulative average of position change for each driver per year
df_unified['Avg_Position_Gain_Championship'] = df_unified.groupby(['DriverId', 'Year'])['Position_Gain_From_Grid'].expanding().mean().reset_index(level=[0, 1], drop=True)

In [ ]:
# El CSV ya trae los tiempos de qualy calculados en silver:
#   Q1_seconds_ABS, Q2_seconds_ABS, Q3_seconds_ABS, Q_Best_seconds_ABS
#   y sus distancias porcentuales al mejor (sufijo _DBT_%).


## 3. PCA de tiempos de vuelta en prácticas

In [ ]:
# 1. Seleccionar todas las columnas relacionadas con FP_LapTime_min y FP_LapTime_mean
# Excluir las columnas `_DBT_%`, ya que la PCA se aplica a los valores absolutos (`_ABS`).
explicit_fp_laptime_cols = [
    'FP_LapTime_min_SOFT_ABS',
    'FP_LapTime_min_MED_ABS',
    'FP_LapTime_min_HARD_ABS',
    'FP_LapTime_min_INTER_ABS',
    'FP_LapTime_min_WET_ABS',
    'FP_LapTime_mean_SOFT_ABS',
    'FP_LapTime_mean_MED_ABS',
]

# Filtrar solo las columnas que realmente existen en el DataFrame
cols_laptime_fp = [col for col in explicit_fp_laptime_cols if col in df_unified.columns]

# Filtrar las columnas numéricas que no sean totalmente NaN
cols_to_combine = [col for col in cols_laptime_fp if df_unified[col].dtype in ['float64', 'int64'] and not df_unified[col].isnull().all()]

if not cols_to_combine:
    print(f"No se encontraron columnas de LapTime de FP válidas para combinar. Columnas intentadas: {explicit_fp_laptime_cols}")
    print(f"Columnas presentes en df_unified: {df_unified.columns.tolist()}")
else:
    print(f"Columnas FP_LapTime a combinar: {cols_to_combine}")

    # 2. Es IMPRESCINDIBLE escalar las variables antes de PCA
    # Rellenar los NaNs con la media de cada columna antes de escalar
    df_temp_pca = df_unified[cols_to_combine].fillna(df_unified[cols_to_combine].mean())

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_temp_pca)

    # 3. Aplicar PCA configurado para extraer exactamente 1 componente
    pca = PCA(n_components=1)
    df_unified['FP_LapTime_Combined_PCA'] = pca.fit_transform(scaled_data)

    # 4. (Opcional) Verificar cuánta información/varianza retuvimos
    print(f"Varianza explicada por el componente principal: {pca.explained_variance_ratio_[0]:.2%}")

    # 5. Eliminar las columnas originales combinadas por PCA
    df_unified = df_unified.drop(columns=cols_to_combine)

    print("\nDataFrame después de PCA y eliminación de columnas originales:")
    df_unified.info()

In [ ]:
# Calculate 'diff_to_best' for FP_LapTime_Combined_PCA
best_pca_in_race = df_unified.groupby(['Year', 'RoundNumber'])['FP_LapTime_Combined_PCA'].transform('min')

df_unified['FP_LapTime_Combined_PCA_diff_to_best'] = df_unified['FP_LapTime_Combined_PCA'] - best_pca_in_race

print("DataFrame después de agregar 'FP_LapTime_Combined_PCA_diff_to_best':")
df_unified.info()

## 4. Grupos de clasificación (Qualy)

In [ ]:
# Crear la variable categórica de tramos de Qualy
condiciones = [
    df_unified['Q_Position'] <= 10,
    (df_unified['Q_Position'] > 10) & (df_unified['Q_Position'] <= 15),
    df_unified['Q_Position'] > 15,
]

etiquetas = ['Top 10 (Q3)', 'Midfield (P11-P15 / Q2)', 'Backmarkers (P16+ / Q1)']

df_unified['Qualy_Group'] = np.select(condiciones, etiquetas, default='Otro')

# Convertir a Categorical para mantener el orden lógico en gráficos y tablas
df_unified['Qualy_Group'] = pd.Categorical(
    df_unified['Qualy_Group'],
    categories=etiquetas,
    ordered=True,
)

## 5. Helpers de correlación y semáforo

In [ ]:
CORTES = {
    "separacion": (0.2, 0.8),
    "eta2": (0.05, 0.25),
    "correlacion": (0.2, 0.6),
    "brecha": (0.05, 0.15),
}

def semaforo(valor, medida):
    """Devuelve la zona en la que cae el valor, según la medida usada."""
    if pd.isna(valor):
        return "sin_datos"
    bajo, alto = CORTES[medida]
    v = abs(valor)
    return "rojo" if v < bajo else "verde" if v > alto else "amarillo"

In [ ]:
def asociacion(df, col_y, col_x):
    # Eliminar filas con NaN en cualquiera de las dos columnas para el cálculo
    df_clean = df[[col_x, col_y]].dropna()

    # Calcular Pearson
    pearson_corr = df_clean[col_x].corr(df_clean[col_y], method='pearson')

    # Calcular Spearman
    spearman_corr = df_clean[col_x].corr(df_clean[col_y], method='spearman')

    # Calcular la 'brecha' o diferencia absoluta entre Pearson y Spearman
    brecha = abs(pearson_corr - spearman_corr)

    return pearson_corr, spearman_corr, brecha

## 6. Correlación con RACE_Position

In [ ]:
# Get all numerical columns in df_unified
numerical_cols = df_unified.select_dtypes(include=np.number).columns.tolist()

excluded_cols_for_correlation = [
    'RACE_Position',
]

# Filter out excluded columns from the numerical list
features_for_correlation = [col for col in numerical_cols if col not in excluded_cols_for_correlation]

# Create a temporary DataFrame with only the relevant columns for correlation
df_corr = df_unified[features_for_correlation + ['RACE_Position']]

# Calculate the correlation of each feature with 'RACE_Position'
correlations = df_corr.corr()['RACE_Position'].drop('RACE_Position')

# Sort correlations by absolute value in descending order
correlations_sorted = correlations.abs().sort_values(ascending=False)

print("Correlations with RACE_Position (absolute values, sorted descending):\n")
print(correlations[correlations_sorted.index])

In [ ]:
# Apply the semaforo function to the absolute correlations
correlation_classification = correlations_sorted.apply(lambda x: semaforo(x, 'correlacion'))

# Display the correlations and their classification
classified_correlations_df = pd.DataFrame({
    'Correlation': correlations[correlations_sorted.index],
    'Absolute_Correlation': correlations_sorted,
    'Classification': correlation_classification,
})

display(classified_correlations_df)

## 7. Análisis Pearson / Spearman / brecha

In [ ]:
# Obtener todas las columnas numéricas en df_unified
numerical_cols_all = df_unified.select_dtypes(include=np.number).columns.tolist()

# Columnas a excluir (resultados de carrera y 'RACE_Position' como variable dependiente)
excluded_cols_for_all_correlation = [
    'RACE_Position',
]

# Filtrar las columnas numéricas para excluir las que no queremos evaluar como predictores
features_to_evaluate_all = [col for col in numerical_cols_all if col not in excluded_cols_for_all_correlation]

# Lista para almacenar los resultados
results_all = []

# Iterar sobre cada característica y aplicar la función asociacion
for feature in features_to_evaluate_all:
    pearson, spearman, brecha = asociacion(df_unified, 'RACE_Position', feature)
    results_all.append({
        'Feature': feature,
        'Pearson': pearson,
        'Spearman': spearman,
        'Brecha': brecha,
        'Semaforo_Pearson': semaforo(pearson, 'correlacion'),
        'Semaforo_Spearman': semaforo(spearman, 'correlacion'),
        'Semaforo_Brecha': semaforo(brecha, 'brecha'),
    })

# Crear un DataFrame con los resultados y ordenarlo por correlación de Pearson descendente
results_df_all = pd.DataFrame(results_all)
display(results_df_all.sort_values(by='Pearson', ascending=False).reset_index(drop=True))

In [ ]:
# Definir las columnas del semáforo
semaforo_cols = ['Semaforo_Pearson', 'Semaforo_Spearman', 'Semaforo_Brecha']

# Función para contar los colores del semáforo en cada fila
def count_semaforo_colors(row):
    verde_count = sum(1 for col in semaforo_cols if row[col] == 'verde')
    amarillo_count = sum(1 for col in semaforo_cols if row[col] == 'amarillo')
    rojo_count = sum(1 for col in semaforo_cols if row[col] == 'rojo')
    return verde_count, amarillo_count, rojo_count

# Aplicar la función y crear las nuevas columnas en results_df_all
results_df_all[['Verde_Count', 'Amarillo_Count', 'Rojo_Count']] = results_df_all.apply(count_semaforo_colors, axis=1, result_type='expand')

# Ordenar: primero más verdes, luego más amarillos, finalmente menos rojos
results_df_sorted = results_df_all.sort_values(by=['Verde_Count', 'Amarillo_Count', 'Rojo_Count'], ascending=[False, False, False]).reset_index(drop=True)

display(results_df_sorted)

In [ ]:
# Crear un DataFrame con solo las características a evaluar
df_features_for_matrix = df_unified[features_to_evaluate_all].copy()

# Calcular la matriz de correlación
correlation_matrix = df_features_for_matrix.corr()

print("\nMatriz de Correlación Cruzada (Pearson):\n")
display(correlation_matrix)

# Visualizar la matriz de correlación con un mapa de calor
plt.figure(figsize=(20, 18))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', fmt=".2f", linewidths=.05)
plt.title('Matriz de Correlación Cruzada de Características (Pearson)')
plt.show()

## 8. Análisis bivariado por grupo de clasificación

In [ ]:
# Resumen estadístico por grupo de Qualy
bivariado_summary = df_unified.groupby('Qualy_Group', observed=False).agg(
    Cant_Casos=('RACE_Position', 'count'),
    Race_Pos_Promedio=('RACE_Position', 'mean'),
    Race_Pos_Mediana=('RACE_Position', 'median'),
    Qualy_Diff_Promedio=('Q_Best_seconds_DBT_%', 'mean'),
    Qualy_Diff_Mediana=('Q_Best_seconds_DBT_%', 'median'),
    Qualy_Diff_Std=('Q_Best_seconds_DBT_%', 'std'),
).reset_index()

display(bivariado_summary)

In [ ]:
# Correlación y brecha entre Qualy y posición en carrera, por grupo
bivariado_resultados = []

for grupo, sub_df in df_unified.groupby('Qualy_Group', observed=False):
    valid = sub_df.dropna(subset=['Q_Best_seconds_DBT_%', 'RACE_Position', 'Q_Position'])

    if len(valid) > 2:
        # Correlación y brecha entre Tiempo de Qualy y Posición en Carrera
        p_tiempo, s_tiempo, brecha_val = asociacion(valid, 'RACE_Position', 'Q_Best_seconds_DBT_%')

        # Correlación entre Posición de Qualy y Posición en Carrera
        p_pos, s_pos, brecha_pos = asociacion(valid, 'RACE_Position', 'Q_Position')

        bivariado_resultados.append({
            'Grupo_Qualy': grupo,
            'Muestra': len(valid),
            'Pearson_Tiempo': round(p_tiempo, 3),
            'Semaforo_P_Tiempo': semaforo(p_tiempo, 'correlacion'),
            'Spearman_Tiempo': round(s_tiempo, 3),
            'Semaforo_S_Tiempo': semaforo(s_tiempo, 'correlacion'),
            'Spearman_Posicion': round(s_pos, 3),
            'Semaforo_S_Posicion': semaforo(s_pos, 'correlacion'),
            'Brecha_Tiempo': round(brecha_val, 3),
            'Semaforo_Brecha': semaforo(brecha_val, 'brecha'),
        })

results_bivariado_df = pd.DataFrame(bivariado_resultados)
display(results_bivariado_df)